In [1]:
import pandas as pd
import massbalancemachine as mbm
import warnings
import geopandas as gpd
import numpy as np
from cmcrameri import cm
import matplotlib.pyplot as plt
from massbalancemachine.data_processing.create_glacier_grid import create_glacier_grid_RGI
from massbalancemachine.data_processing.get_topo_data import get_glacier_mask
from joblib import load

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'massbalancemachine.data_processing.create_glacier_grid'

In [ ]:
# Especifique el nombre del archivo de entrada que contenga los ID RGIId
glas_fname = './data/SouthernAndes_topographical_features.csv'
# Carge el archivo
data = pd.read_csv(glas_fname)
cfg = mbm.Config()
display(data)

In [ ]:
rgi_gl = data.RGIId.unique()
print(rgi_gl)

In [ ]:
data1 = data[data.RGIId==rgi_gl[1]]
# Definimos el directorio donde están los datos de OGGM
custom_working_dir='OGGM/'
ds, glacier_indices, gdir = get_glacier_mask(data1, custom_working_dir=custom_working_dir, cfg=cfg)

In [ ]:
# Definimos el periodo, para este ejemplo solo usaremos 1 año
years = range(2016, 2017)
# Extraemos el dataframe 
df_grid = create_glacier_grid_RGI(ds, years, glacier_indices, gdir,
                                rgi_gl[0])

In [ ]:
# Mantenemos variables necesarias 
var_topo = ['YEAR', 'RGIId', 'POINT_LAT', 'POINT_LON', 'aspect', 'slope',
            'POINT_ELEVATION', 'POINT_BALANCE',
            'FROM_DATE', 'TO_DATE']
data_glas =  df_grid[var_topo]
data_glas['POINT_ID'] = data_glas['RGIId']

In [ ]:
def plotGlAttr(ds, cmap=cm.batlow):
    # Plot glacier attributes
    fig, ax = plt.subplots(2, 3, figsize=(18, 10))
    ds.masked_slope.plot(ax=ax[0, 0], cmap=cmap)
    ax[0, 0].set_title('Slope')
    ds.masked_elev.plot(ax=ax[0, 1], cmap=cmap)
    ax[0, 1].set_title('Elevation')
    ds.masked_aspect.plot(ax=ax[0, 2], cmap=cmap)
    ax[0, 2].set_title('Aspect')
    ds.masked_hug.plot(ax=ax[1, 0], cmap=cmap)
    ax[1, 0].set_title('Hugonnet')
    ds.masked_cit.plot(ax=ax[1, 1], cmap=cmap)
    ax[1, 1].set_title('Consensus ice thickness')
    ds.masked_miv.plot(ax=ax[1, 2], cmap=cmap)
    ax[1, 2].set_title('Millan v')
    plt.tight_layout()

In [ ]:
plotGlAttr(ds, cmap=cm.devon)

In [ ]:
# Proporcione el nombre de la columna que contiene los ID de RGI para el glacir
dataset = mbm.data_processing.Dataset(cfg=cfg, data=data_glas, region_name='schiaparelli_grid_2013', data_path='./grid/')

In [ ]:
# Especifique los archivos de datos climáticos que se corresponderán con las coordenadas de los datos de estaca
era5_climate_data = './era5land/raw/era5_monthly_averaged_data.nc'
geopotential_data = '../../notebooks/example_data/iceland/climate/era5_geopotential_pressure.nc'

# Haga coincidir las características climáticas, del archivo netCDF de ERA5Land, para cada conjunto de datos de medición de estaca
dataset.get_climate_features(climate_data=era5_climate_data, geopotential_data=geopotential_data) 


In [ ]:
# Especifique los nombres cortos de las variables climáticas disponibles en el conjunto de datos
vois_climate = ['t2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str']
voi_topographical = ['aspect', 'slope']
# Para cada registro, conviértalo a una resolución de tiempo mensual
dataset.convert_to_monthly(vois_climate=vois_climate, vois_topographical=voi_topographical)


In [ ]:
# Visualizando 
display(dataset.data)

In [ ]:
XGB_model = load('./models/model_xgb.pkl')
data = dataset.data.copy()

In [ ]:
nn = XGB_model.best_estimator_.set_params(device='cpu')

features, metadata = nn._create_features_metadata(data)

feature_columns = data.columns.difference(cfg.metaData)
feature_columns = feature_columns.drop(cfg.notMetaDataNotFeatures)
feature_columns = list(feature_columns)

bounds_features = {k:(np.min(data[k].values), np.max(data[k].values)) for k in feature_columns}
norm = mbm.data_processing.Normalizer(bounds_features)
norm_features = norm.normalize(features)

norm_features

In [ ]:
data['MB'] = nn.predict(norm_features)

In [ ]:
data["MONTHS"]

## Calculo Anual

In [ ]:
##Agrupar predicciones mensuales (en un promedio) segun una coordenada
data_anual = data.groupby(['POINT_LAT', 'POINT_LON', 'ID', 'YEAR'], as_index=False)['MB'].mean()
data_anual.rename(columns={'MB': 'MB_ANUAL_PROMEDIO'}, inplace=True)
data_anual.head(15)

In [ ]:
# Carga de la mask de OGGM
glas_fname = './data/SouthernAndes_topographical_features.csv'
data = pd.read_csv(glas_fname)
cfg = mbm.Config()

In [ ]:
#Extraer predicciones y moverlas al mask del glaciar
def prediction_anual_to_mask(ds, year, data_anual):
    glacier_mask = np.where(ds['glacier_mask'].values == 0, np.nan,
                            ds['glacier_mask'].values)
    mask = glacier_mask == 1.0
    aux_data = data_anual[ data_anual['YEAR'] == year ]
    valores = aux_data['MB_ANUAL_PROMEDIO'].reset_index(drop=True)

    flat_vals = pd.Series(valores, index=np.argwhere(mask).tolist())
    nueva_matriz = glacier_mask.copy()
    idxs = np.argwhere(mask)
    for i, (row, col) in enumerate(idxs):
        nueva_matriz[row, col] = valores[i]
    return nueva_matriz

In [ ]:
anio = 2016
glaciar = "Grey + Dickson"

fig, ax = plt.subplots(figsize=(8,6))
plt.imshow( prediction_to_mask(ds, anio, data_anual) , cmap='cividis', interpolation='nearest')
plt.colorbar(label="(m.eq.w)")
plt.title(f"Prediccion balance de masa anual {anio} glaciar {glaciar}")
plt.savefig(f"bm anual-{glaciar}-{anio}.png", dpi=300, bbox_inches="tight")
plt.show()

## Calculo mensual

In [ ]:
##Agrupar predicciones mensuales (en un promedio) segun una coordenada
data_mensual = data.copy()#.groupby(['POINT_LAT', 'POINT_LON', 'ID', 'YEAR'], as_index=False)

In [ ]:
#Extraer predicciones y moverlas al mask del glaciar
def prediction_year_mensual_to_mask(ds, year, month, data_mensual):
    glacier_mask = np.where(ds['glacier_mask'].values == 0, np.nan,
                            ds['glacier_mask'].values)
    mask = glacier_mask == 1.0
    aux_data = data_mensual[ (data_mensual['YEAR'] == year) & (data_mensual['MONTHS'] == month) ]
    valores = aux_data['MB'].reset_index(drop=True)

    flat_vals = pd.Series(valores, index=np.argwhere(mask).tolist())
    nueva_matriz = glacier_mask.copy()
    idxs = np.argwhere(mask)
    for i, (row, col) in enumerate(idxs):
        nueva_matriz[row, col] = valores[i]
    return nueva_matriz

In [ ]:
anio = 2016
month = "jan"
glaciar = "Grey + Dickson"

fig, ax = plt.subplots(figsize=(8,6))
plt.imshow( prediction_year_mensual_to_mask(ds, anio, month, data_mensual) , cmap='cividis', interpolation='nearest')
plt.colorbar(label="(m.eq.w)")
plt.title(f"Prediccion balance de masa mensual {month}-{anio} glaciar {glaciar}")
plt.savefig(f"bm mensual-{glaciar}-{anio}-{month}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
anio = 2016
month = "may"
glaciar = "Grey + Dickson"

fig, ax = plt.subplots(figsize=(8,6))
plt.imshow( prediction_year_mensual_to_mask(ds, anio, month, data_mensual) , cmap='cividis', interpolation='nearest')
plt.colorbar(label="(m.eq.w)")
plt.title(f"Prediccion balance de masa mensual {month}-{anio} glaciar {glaciar}")
plt.savefig(f"bm mensual-{glaciar}-{anio}-{month}.png", dpi=300, bbox_inches="tight")
plt.show()